<a href="https://colab.research.google.com/github/dtoralg/TheValley_MDS/blob/main/%5B02%5D%20-%20Analisis_Cluster/%5B01%5D%20-%20Notebooks/E2_Dendrograma_y_Corte_Iris.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E2 · Dendrograma y corte - Análisis cluster

## Introducción

¿Y si no quieres fijar K de antemano? El **clustering jerárquico** construye una jerarquía de
grupos uniendo, paso a paso, las observaciones más cercanas (estrategia **aglomerativa**,
bottom-up). El resultado se ve en un **dendrograma**: un árbol de fusiones donde la **altura del
corte** decide cuántos grupos obtienes.

La clave es el **linkage** (cómo se mide la distancia entre grupos). Usamos **Ward**, que suele
ser un buen punto de partida. Dibujamos el dendrograma y probamos cortes en 2, 3 y 4 grupos.

## Objetivos del ejercicio

- Entender el **dendrograma** y el papel del **linkage** (Ward).
- Cortar el árbol en 2, 3 y 4 grupos y comparar.
- Elegir un corte y comprobarlo con las especies reales del dataset.

## Descripción del dataset (Iris)

**Iris** es un dataset real clásico: 150 flores de 3 especies, con 4 medidas (largo y ancho de
sépalo y pétalo). Es pequeño e ideal para ver un dendrograma claro. Viene incluido en sklearn.

### 1. Importar librerías necesarias

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

### 2. Cargar Iris y escalar

In [ ]:
iris = load_iris(as_frame=True)
X = iris.data
y = iris.target          # especie real (solo para comprobar al final)
print("Forma:", X.shape, "| especies:", list(iris.target_names))

X_esc = StandardScaler().fit_transform(X)

### 3. Linkage 'ward' y dendrograma

In [ ]:
Z = linkage(X_esc, method="ward")     # matriz de fusiones del clustering jerárquico

plt.figure(figsize=(11, 5))
dendrogram(Z, no_labels=True, color_threshold=7)
plt.title("Dendrograma (linkage = ward)")
plt.xlabel("Observaciones")
plt.ylabel("Distancia de fusión")
plt.tight_layout(); plt.show()

### 4. Probar cortes en 2, 3 y 4 grupos

`fcluster` corta el árbol para obtener el número de grupos que pidamos. Vemos cuántos clientes
caen en cada grupo según el corte.

In [ ]:
for k in [2, 3, 4]:
    labels = fcluster(Z, t=k, criterion="maxclust")
    tam = np.bincount(labels)[1:]    # fcluster etiqueta desde 1
    print(f"Corte en {k} grupos -> tamaños: {tam}")

### 5. ¿Qué corte elegimos? Comprobación con las especies reales

In [ ]:
labels3 = fcluster(Z, t=3, criterion="maxclust")
tabla = pd.crosstab(labels3, iris.target_names[y], rownames=["cluster"], colnames=["especie"])
print("Cruce del corte en 3 grupos con la especie real:")
print(tabla)
print(f"\nAdjusted Rand Index (3 grupos vs especie real): {adjusted_rand_score(y, labels3):.3f}")
print("El corte en 3 reconstruye bastante bien las 3 especies de iris.")

### Output: dendrograma + corte elegido

Elegimos el corte en **3 grupos**: es el que sugiere visualmente el dendrograma (las tres ramas
grandes) y el que mejor coincide con las especies reales.

### Reflexión

1. ¿Por qué el clustering jerárquico no te obliga a fijar K antes de empezar?
2. ¿Cómo decides la altura del corte mirando el dendrograma?
3. ¿Qué dos especies de iris se confunden más entre sí?
4. ¿Qué ventaja y qué inconveniente tiene el jerárquico frente a K-Means?